In [6]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# Step 1: Load Data
customers_df = pd.read_csv('customer_data.csv')  # Assuming columns: FUND_ID, ASSET_CLASS_ID, CUSTOMER_ID, AVAILABLE_AMOUNT
funds_df = pd.read_csv('fund_allocation.csv')   # Assuming columns: FUND_ID, ASSET_CLASS_ID, PERCENT_OF_FUND

# Step 2: Define the objective function to minimize the difference between available amount and allocation required
def objective_function(limits, customers_df, funds_df, fund_id):
    min_amount, max_amount = limits
    
    # Calculate the sum of squared differences between available amount and allocated amount
    diff_sum = 0
    for i, row in customers_df.iterrows():
        # Find fund's allocation percentage for the corresponding asset class
        fund_allocation = funds_df[(funds_df['FUND_ID'] == fund_id) & (funds_df['ASSET_CLASS_ID'] == row['ASSET_CLASS_ID'])]
        
        if not fund_allocation.empty:
            # Required amount for this asset class based on fund strategy
            required_amount = max_amount * fund_allocation['PERCENT_OF_FUND'].values[0]
            
            # Penalize if customer's available amount is less than the minimum or more than the required amount
            if row['AVAILABLE_AMOUNT'] < min_amount:
                diff_sum += (min_amount - row['AVAILABLE_AMOUNT']) ** 2
            elif row['AVAILABLE_AMOUNT'] > required_amount:
                diff_sum += (row['AVAILABLE_AMOUNT'] - required_amount) ** 2

    return diff_sum  # Minimize this difference

# Step 3: Define constraints
def constraint(limits, customers_df, funds_df, fund_id):
    min_amount, max_amount = limits
    # Ensure that the max amount is greater than or equal to the min amount
    return max_amount - min_amount

# Step 4: Optimization
def optimize_investment_limits(customers_df, funds_df, fund_id):
    # Initial guess for the minimum and maximum amounts
    initial_guess = [10000, 100000]  # Initial guesses for min and max amounts (arbitrary)

    # Set bounds for min and max amounts (min cannot be negative, max can be a large number)
    bounds = [(0, None), (0, None)]

    # Set the constraint that the max amount must be greater than or equal to the min amount
    cons = {'type': 'ineq', 'fun': lambda limits: constraint(limits, customers_df, funds_df, fund_id)}

    # Perform the optimization
    result = minimize(objective_function, initial_guess, args=(customers_df, funds_df, fund_id), 
                      method='SLSQP', bounds=bounds, constraints=cons)

    return result.x




In [7]:
results_df = pd.DataFrame(columns=['FUND_ID', 'MIN_AMOUNT', 'MAX_AMOUNT', 'TOTAL_CUSTOMERS', 'ELIGIBLE_CUSTOMERS', 'ELIGIBILITY_PERCENTAGE'])

# Function to determine customer eligibility based on optimized limits
def calculate_eligibility(customers_df, limits, funds_df, fund_id):
    min_amount, max_amount = limits
    
    eligible_count = 0
    total_count = len(customers_df)  # Count of customers for this specific fund
    
    for _, row in customers_df.iterrows():
        # Find fund's allocation percentage for the corresponding asset class
        fund_allocation = funds_df[(funds_df['FUND_ID'] == fund_id) & (funds_df['ASSET_CLASS_ID'] == row['ASSET_CLASS_ID'])]
        
        if not fund_allocation.empty:
            # Check if the customer's available amount falls within the min and max limits
            if min_amount <= row['AVAILABLE_AMOUNT'] <= (max_amount * fund_allocation['PERCENT_OF_FUND'].values[0]):
                eligible_count += 1

    eligibility_percentage = (eligible_count / total_count) * 100 if total_count > 0 else 0
    
    return eligible_count, total_count, eligibility_percentage

# Step 5: Run the optimization and calculate eligibility for each fund
fund_ids = customers_df['FUND_ID'].unique()

for fund_id in fund_ids:
    # Filter customers for the current fund
    fund_customers = customers_df[customers_df['FUND_ID'] == fund_id]
    
    # Get optimized limits for the current fund
    optimized_limits = optimize_investment_limits(fund_customers, funds_df, fund_id)
    min_amount, max_amount = optimized_limits
    
    # Calculate eligibility
    eligible_count, total_count, eligibility_percentage = calculate_eligibility(
        fund_customers, optimized_limits, funds_df, fund_id)
    
    fund_results = pd.DataFrame([{
        'FUND_ID': fund_id,
        'MIN_AMOUNT': min_amount,
        'MAX_AMOUNT': max_amount,
        'TOTAL_CUSTOMERS': total_count/8,
        'ELIGIBLE_CUSTOMERS': eligible_count/8,
        'ELIGIBILITY_PERCENTAGE': eligibility_percentage
    }])

    # Concatenate the results DataFrame
    results_df = pd.concat([results_df, fund_results], ignore_index=True)

# Step 6: Save results to a CSV file for further use
results_df.to_csv('fund_investment_limits_results.csv', index=False)

print("Results saved to 'fund_investment_limits_results.csv'.")

Results saved to 'fund_investment_limits_results.csv'.
